# ABS labour force — download, validate and tidy

The notebook uses the reference workbooks and the latest completed output to calculate the next release month, builds three ABS download URLs, checks the downloaded workbooks against the reference workbooks, and creates three cleaned tables, one for each source workbook.

**Outputs:** three cleaned CSV files named for their source table and reference month, plus a source manifest, in a new dated folder under `outputs/abs_labour_force/`. Downloads are retained under `raw/` in that folder. The source manifest is written only after all three clean files are saved; earlier successful runs are kept separately.

Keep the three reviewed reference workbooks in **`Labour Force Datasets/`**: they are the structural baseline, not a download destination. When you intentionally replace them with a newer release, review the new structure and update their saved SHA-256 fingerprints in section 1. The run then starts from the later of the reference month and the last completed output month.

### What is a material change?

| Stop the run | Allow without changing the baseline |
|---|---|
| Missing or additional sheets; moved header rows; changed table identity | New monthly observations |
| Added, removed or duplicate series IDs; changed descriptions, units, frequency, series type, or historical start | Revisions to numeric estimates |
| Changed MLF1 columns, age/sex categories, or region codes/names (including geography vintage) | Reordered series columns within a data sheet; reordered MLF1 rows |
| Blank or invalid estimates in MLF1 | Cosmetic formatting; copyright and release-date updates |

Checks are ordinary assertions and pandas comparisons. There are **no exception handlers, retries, fallbacks or automatic baseline updates**. A failed check leaves its traceback visible. The time-series checks focus on structure; MLF1 also checks for blank or non-numeric estimates. These checks cannot certify an ABS methodology change that leaves the metadata unchanged. Review the ABS release notes as well.

## 1. Setup and fixed baseline

Use Python 3.10+ with `pandas`, `numpy`, `openpyxl`, `requests`, `beautifulsoup4`, `pyarrow` and a Jupyter kernel. Install missing packages in your notebook environment with `%pip install pandas numpy openpyxl requests beautifulsoup4 pyarrow` once, then restart the kernel.

Imports appear once so this notebook also works in a fresh kernel. All subsequent code is linear. Paths assume the notebook is opened from the project folder.

In [1]:
%pip install pandas numpy openpyxl requests beautifulsoup4 pyarrow


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from hashlib import sha256
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import requests
from IPython.display import display

root = Path.cwd()
baseline = root / 'Labour Force Datasets'
retrieved_at = datetime.now(timezone.utc).isoformat()
run = root / 'outputs' / 'abs_labour_force' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')

In [3]:
# Identify the reference workbooks and confirm they have not changed before downloading a new ABS release.

# This list pairs each reference workbook with its ABS filename and saved fingerprint.

sources = pd.DataFrame([
    ('Table 010', 'Table 010.xlsx', '62020010.xlsx', 'a03639c10c42acd736ccc24df5680d25d4317720ca724eb2008e0527f729bf45'),
    ('Table 014', 'Interim Table 14.xlsx', '62020014.xlsx', '517c88d31a571408e3a8caf09138e7c205d6232115a17231dd111dabb550e39f'),
    ('MLF1', 'MLF1.xlsx', 'MLF1.xlsx', 'e228ca0d7b4856aea4ef0e273b895c8e836afd6df07b0dae5c814565290d4426')
], columns=['dataset', 'baseline_file', 'download_file', 'baseline_sha256']).set_index('dataset')


# SHA-256 reads every byte of each reference workbook to calculate its fingerprint; the run stops if it differs from the saved fingerprint.

for row in sources.itertuples():
    assert sha256((baseline / row.baseline_file).read_bytes()).hexdigest() == row.baseline_sha256, f'{row.Index}: baseline file changed; review before changing its hash'

# Show the reference and ABS filenames so they can be checked at a glance.

display(sources[['baseline_file', 'download_file']])


,baseline_file,download_file
dataset,,
Table 010,Table 010.xlsx,62020010.xlsx
Table 014,Interim Table 14.xlsx,62020014.xlsx
MLF1,MLF1.xlsx,MLF1.xlsx


## 2. Build the next release URLs and download all three files

Read the last month in the reference Table 010 workbook and in any completed output. The next release is one month after the later of those dates. For the three workbooks, build the observed ABS URL pattern from that month and each fixed filename. The download fails if a file is unavailable. MLF1 is scheduled to move to a separate ABS publication from October 2026, so its URL will need updating then.


In [4]:
# Select the next ABS release, download its three workbooks, and show the chosen sources.
reference_file = baseline / sources.loc['Table 010', 'baseline_file']  # Locate the approved Table 010 workbook.
reference_data = pd.read_excel(reference_file, sheet_name='Data1', header=None)  # Read its monthly data sheet.
reference_month = pd.to_datetime(reference_data.iloc[-1, 0])  # Take the final month in that workbook.
completed = sorted((root / 'outputs' / 'abs_labour_force').glob('*/sources.csv'))  # Find completed prior runs.
processed_months = [reference_month]  # Start with the approved reference month.
for path in completed:  # Consider the latest month in each completed run.
    processed_months.append(pd.to_datetime(pd.read_csv(path)['reference_period']).max())  # Add that run's final month.
reference_period = max(processed_months) + pd.offsets.MonthBegin(1)  # Target the month after the latest one processed.

release_folder = reference_period.strftime('%b-%Y').lower()  # Format the month as ABS uses it in URLs.
site = 'https://www.abs.gov.au/statistics/labour/employment-and-unemployment/'  # Base address for ABS labour releases.
release_url = f'{site}labour-force-australia/{release_folder}/'  # Point to the target month's release folder.
sources['source_url'] = release_url + sources['download_file']  # Add each expected workbook filename.
sources['reference_period'] = reference_period  # Record the month this run is downloading.
sources['retrieved_at_utc'] = retrieved_at  # Record when this run started.

raw_dir = run / 'raw'  # Keep the original downloads in this run's folder.
raw_dir.mkdir(parents=True)  # Create that folder before downloading.
for row in sources.itertuples():  # Download each of the three workbooks.
    r = requests.get(row.source_url, timeout=120, headers={'Cache-Control': 'no-cache'})  # Request the calculated ABS URL.
    r.raise_for_status()  # Stop if ABS did not return a successful response.
    assert r.content.startswith(b'PK'), f'{row.Index}: download is not an XLSX file'  # Check for an Excel file.
    (raw_dir / row.download_file).write_bytes(r.content)  # Save the unmodified ABS download.
    sources.loc[row.Index, 'download_sha256'] = sha256(r.content).hexdigest()  # Fingerprint the downloaded bytes.

print('ABS reference month:', reference_period.strftime('%B %Y'))  # Show which month was downloaded.
display(sources[['download_file', 'reference_period', 'source_url']])  # Preview the selected files and URLs.


ABS reference month: August 2026


,download_file,reference_period,source_url
dataset,,,
Table 010,62020010.xlsx,2026-08-01,https://www.abs.gov.au/statistics/labour/emplo...
Table 014,62020014.xlsx,2026-08-01,https://www.abs.gov.au/statistics/labour/emplo...
MLF1,MLF1.xlsx,2026-08-01,https://www.abs.gov.au/statistics/labour/emplo...


## 3. Read the time-series workbooks

Tables 010 and 014 share an ABS time-series layout: ten metadata rows followed by monthly values. Read every populated data column. Expected sheets, header labels and table titles come from the fixed baseline, rather than from the new download.

In [5]:
# Read the two reference and downloaded time-series workbooks, compare their basic layout, and preview their sizes.
base_books = {}  # Hold every sheet from each approved reference workbook.
books = {}  # Hold every sheet from each newly downloaded workbook.
for dataset in ['Table 010', 'Table 014']:  # Repeat these checks for the two time-series tables.
    row = sources.loc[dataset]  # Get this table's reference and download filenames.
    base_books[dataset] = pd.read_excel(baseline / row['baseline_file'], sheet_name=None, header=None)  # Read all reference sheets.
    books[dataset] = pd.read_excel(run / 'raw' / row['download_file'], sheet_name=None, header=None)  # Read all downloaded sheets.
    assert set(books[dataset]) == set(base_books[dataset]), f'{dataset}: workbook sheets changed'  # Require the same sheet names.
    reference_title = base_books[dataset]['Index'].iloc[4:6, 1]  # Get the approved publication and table titles.
    downloaded_title = books[dataset]['Index'].iloc[4:6, 1]  # Get those titles from the new workbook.
    pd.testing.assert_series_equal(downloaded_title, reference_title, check_names=False, obj=f'{dataset}: publication and table title')  # Require identical titles.

summary = []  # Collect a quick size check for each downloaded data sheet.
for dataset, book in books.items():  # Visit each downloaded workbook.
    for sheet, sheet_data in book.items():  # Visit each sheet in that workbook.
        if sheet.startswith('Data'):  # Include only sheets containing monthly series.
            summary.append((dataset, sheet, sheet_data.shape[0] - 10, sheet_data.shape[1] - 1))  # Count months and series.
display(pd.DataFrame(summary, columns=['dataset', 'sheet', 'months', 'series']))  # Show the workbook sizes.


,dataset,sheet,months,series
0,Table 010,Data1,583,250
1,Table 010,Data2,583,250
2,Table 010,Data3,583,244
3,Table 014,Data1,485,90


### Validate series definitions and reshape monthly observations

For each data sheet, the first seven metadata fields must match the reference by ABS series ID. End dates, observation counts, monthly continuity and numeric completeness are not separately checked in this cell. The monthly values are reshaped into one row per date and series for the clean table.


In [6]:
# Check each series definition against the reference and reshape monthly values into tidy rows.
metadata = []  # Set up an empty list to save the series definitions from each data sheet.
observations = []  # Set up an empty list to save the monthly values from each data sheet.
checks = []  # Set up an empty list to save a summary of each data sheet.
labels = ['description', 'unit', 'series_type', 'data_type', 'frequency', 'collection_month', 'series_start', 'series_end', 'n_obs', 'series_id']  # Name the ten metadata fields.

for dataset, book in books.items():  # Check each downloaded time-series workbook and sheet, skip index and other non-data sheets.
    for sheet, raw in book.items():  
        if not sheet.startswith('Data'):
            continue 
        base = base_books[dataset][sheet]  # Get the matching reference sheet.
        pd.testing.assert_series_equal(raw.iloc[:10, 0], base.iloc[:10, 0], obj=f'{dataset}/{sheet}: header rows')  # Require the same header positions.
        assert raw.shape[1] == base.shape[1], f'{dataset}/{sheet}: series column count changed'  # Require the same number of series columns.
        assert raw.iloc[9, 1:].notna().all() and raw.iloc[9, 1:].is_unique, f'{dataset}/{sheet}: missing or duplicate series IDs'  # Require valid, distinct IDs.

        meta = (  # Arrange the new workbook's metadata by series ID.
            raw.iloc[:10, 1:].T  # Put each series on its own row.
               .set_axis(labels, axis=1)  # Name the metadata fields.
               .set_index('series_id')  # Use the stable ABS series ID as the key.
               .sort_index()  # Align series before comparison.
        )
        expected = (  # Arrange the reference metadata in the same way.
            base.iloc[:10, 1:].T  # Put each reference series on its own row.
                .set_axis(labels, axis=1)  # Apply the same field names.
                .set_index('series_id')  # Match by ABS series ID.
                .sort_index()  # Align reference series before comparison.
        )
        pd.testing.assert_frame_equal(meta.iloc[:, :7], expected.iloc[:, :7], obj=f'{dataset}/{sheet}: series definitions')  # Stop if definitions changed.

        df = raw.iloc[10:, :].copy()  # Keep the monthly observations below the metadata.
        df.columns = ['date'] + raw.iloc[9, 1:].tolist()  # Name each value column with its series ID.
        df['date'] = pd.to_datetime(df['date'])  # Treat month labels as dates.

        values = df.set_index('date').infer_objects()  # Keep dates as the row key and infer numeric columns.
        long = values.rename_axis(columns='series_id').reset_index().melt(id_vars='date', value_name='value')  # Make one date/series/value row per estimate.
        observations.append(long.assign(dataset=dataset))  # Add this sheet’s monthly values to the list.
        metadata.append(meta.reset_index().assign(dataset=dataset, sheet=sheet))  # Add this sheet’s series definitions to the list.
        checks.append((dataset, sheet, len(df), len(meta), len(long), df['date'].max()))  # Add this sheet’s counts and latest month to the list.

meta = pd.concat(metadata, ignore_index=True)  # Combine the saved definitions into one DataFrame.
ts = pd.concat(observations, ignore_index=True)  # Combine the saved monthly values into one DataFrame.
assert not meta.duplicated(['dataset', 'series_id']).any(), 'A series ID is repeated across sheets'  # Require unique IDs within each table.
display(pd.DataFrame(checks, columns=['dataset', 'sheet', 'months', 'series', 'observations', 'latest_month']))  # Preview the validation counts.


/var/folders/kb/9djgs_0j5m10183fr8tt5qzr0000gn/T/ipykernel_25209/4052107100.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  long = values.rename_axis(columns='series_id').reset_index().melt(id_vars='date', value_name='value')  # Make one date/series/value row per estimate.
/var/folders/kb/9djgs_0j5m10183fr8tt5qzr0000gn/T/ipykernel_25209/4052107100.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  long = values.rename_axis(columns='series_id').reset_index().melt(id_vars='date', value_name='value')  # Make one date/series/

,dataset,sheet,months,series,observations,latest_month
0,Table 010,Data1,583,250,145750,2026-08-01
1,Table 010,Data2,583,250,145750,2026-08-01
2,Table 010,Data3,583,244,142252,2026-08-01
3,Table 014,Data1,485,90,43650,2026-08-01


### Split the descriptions into clean dimensions

Assumptions: Table 010 covers people aged 15 years and over; Table 014 covers persons aged 15–24. ABS hierarchy markers (`>`) are removed only after the original descriptions have passed validation. Published units are retained: `000` means thousands of people and `Percent` stays as a percentage: 4.5 means 4.5%, not 0.045.

In [7]:
# Split each ABS series description into dashboard-friendly fields and attach them to the monthly values.
parts = meta['description'].str.strip().str.rstrip(';').str.split(';', expand=True)  # Divide each description at its semicolons.
parts = parts.apply(lambda s: s.str.strip().str.lstrip('>').str.strip())  # Remove spacing and ABS hierarchy markers.
meta['measure'] = parts[0]  # Save what the series measures.
meta['sex'] = parts[1]  # Save the sex category.
meta['region'] = parts[2]  # Save the geographic category.
meta['age'] = '15 years and over'  # Set the usual age group for these tables.
youth = meta['dataset'].eq('Table 014')  # Identify Table 014's youth series.
meta.loc[youth, 'region'] = parts.loc[youth, 0]  # In Table 014, the first part names the region.
meta.loc[youth, 'measure'] = parts.loc[youth, 1]  # In Table 014, the second part names the measure.
meta.loc[youth, 'sex'] = 'Persons'  # Table 014 is not split by sex.
meta.loc[youth, 'age'] = '15-24 years'  # Table 014 covers people aged 15 to 24.
assert meta[['measure', 'sex', 'region', 'age']].notna().all().all()  # Require each new label to be present.

meta['series_start'] = pd.to_datetime(meta['series_start'])  # Preserve the published series start as a date.
meta['series_end'] = pd.to_datetime(meta['series_end'])  # Preserve the published series end as a date.
meta['n_obs'] = pd.to_numeric(meta['n_obs']).astype('Int64')  # Preserve the published observation count.
ts = ts.merge(meta[['dataset', 'series_id', 'measure', 'sex', 'region', 'age', 'unit', 'series_type', 'data_type', 'frequency', 'collection_month', 'series_start', 'series_end', 'n_obs']], on=['dataset', 'series_id'], validate='many_to_one')  # Retain all nine named source metadata fields alongside the clean dimensions.
ts['region_code'] = ts['region'].map({  # Translate region names into stable short codes.
    'Australia': 'AUS', 'New South Wales': '1', 'Victoria': '2', 'Queensland': '3',  # National, NSW, Victoria, and Queensland.
    'South Australia': '4', 'Western Australia': '5', 'Tasmania': '6',  # South Australia, Western Australia, and Tasmania.
    'Northern Territory': '7', 'Australian Capital Territory': '8'  # Northern Territory and ACT.
})
assert ts['region_code'].notna().all()  # Require every region name to have a code.
ts['geography_level'] = np.where(ts['region'].eq('Australia'), 'Australia', 'State/Territory')  # Mark national versus state-level rows.
ts['geography_basis'] = 'Australia / state and territory'  # Record the geographic level used by these tables.
print(ts.shape)  # Show the size of the enriched monthly table.
display(ts[['date', 'series_id', 'measure', 'region', 'sex', 'age', 'series_type', 'data_type', 'unit', 'value']].head())  # Preview its columns and values.


(477402, 19)


,date,series_id,measure,region,sex,age,series_type,data_type,unit,value
0,1978-02-01,A84423127L,Employed total,Australia,Persons,15 years and over,Trend,STOCK,000,6007.747586
1,1978-03-01,A84423127L,Employed total,Australia,Persons,15 years and over,Trend,STOCK,000,6014.722083
2,1978-04-01,A84423127L,Employed total,Australia,Persons,15 years and over,Trend,STOCK,000,6021.174221
3,1978-05-01,A84423127L,Employed total,Australia,Persons,15 years and over,Trend,STOCK,000,6026.885932
4,1978-06-01,A84423127L,Employed total,Australia,Persons,15 years and over,Trend,STOCK,000,6030.961427


## 4. Read and validate MLF1 regional estimates

Use the underlying **Data 1** sheet, which contains all sex × age × region combinations. The presentation pivot sheet is not another dataset and is not appended. Ignore only completely empty columns below the workbook preamble; any populated new header or data column is compared with the baseline.

In [8]:
# Read the MLF1 reference and download, then compare their sheets and table headings.
row = sources.loc['MLF1']  # Get MLF1's reference and download filenames.
base_xl = pd.ExcelFile(baseline / row['baseline_file'])  # Open the approved reference workbook.
xl = pd.ExcelFile(run / 'raw' / row['download_file'])  # Open the new ABS workbook.
assert set(xl.sheet_names) == set(base_xl.sheet_names), 'MLF1: workbook sheets changed'  # Require the same sheets.
base_raw = pd.read_excel(base_xl, sheet_name='Data 1', header=None)  # Read the reference data sheet as it appears in Excel.
raw = pd.read_excel(xl, sheet_name='Data 1', header=None)  # Read the downloaded data sheet the same way.
assert raw.iloc[1, 0] == base_raw.iloc[1, 0], 'MLF1: table identity changed'  # Require the same table title.
expected = base_raw.iloc[3:].dropna(axis=1, how='all')  # Skip the preamble and empty reference columns.
df = raw.iloc[3:].dropna(axis=1, how='all')  # Skip the preamble and empty downloaded columns.
assert df.iloc[0].tolist() == expected.iloc[0].tolist(), 'MLF1: column names, units, order or geography vintage changed'  # Require matching headers.
df.columns = df.iloc[0].tolist()  # Use the downloaded header row as column names.
expected.columns = expected.iloc[0].tolist()  # Name the reference columns the same way.
df = df.iloc[1:].reset_index(drop=True).infer_objects()  # Keep downloaded data rows and infer their types.
expected = expected.iloc[1:].reset_index(drop=True).infer_objects()  # Keep reference data rows for comparison.
assert df.notna().all().all(), 'MLF1: missing dimension or estimate'  # Stop if a downloaded data cell is blank.
print(df.shape)  # Show the size of the downloaded regional table.
display(df.head())  # Preview its first rows.


(183744, 7)


,Month,Sex,Age,Labour market region (SA4): ASGS (2011),Employed total ('000),Unemployed total ('000),Not in the labour force (NILF) ('000)
0,2012-01-01,Males,15-24 years,102 Central Coast,12.757092,2.090347,5.996909
1,2012-01-01,Males,15-24 years,115 Sydney - Baulkham Hills and Hawkesbury,9.284133,1.077604,5.995995
2,2012-01-01,Males,15-24 years,116 Sydney - Blacktown,12.403918,2.240202,8.865894
3,2012-01-01,Males,15-24 years,117 Sydney - City and Inner South,11.087451,1.585265,7.928945
4,2012-01-01,Males,15-24 years,118 Sydney - Eastern Suburbs,8.474928,1.107462,8.446889


In [9]:
# Compare MLF1 categories with the reference and preview the regional dimensions.
dims = df.columns[1:4].tolist()  # Identify sex, age, and region columns.
measures = df.columns[4:].tolist()  # Identify the three estimate columns.
for col in dims:  # Check each category column against the reference.
    assert set(df[col]) == set(expected[col]), f'MLF1: categories changed in {col}'  # Stop if a category was added, removed, or renamed.

df['Month'] = pd.to_datetime(df['Month'])  # Treat the month column as dates.
counts = df.groupby('Month').size()  # Count rows per month for the preview below.
assert df[measures].dtypes.map(pd.api.types.is_numeric_dtype).all(), 'MLF1: non-numeric observations' # Require numeric estimates.

display(pd.DataFrame({  # Preview the regional table's coverage.
    'months': [len(counts)], 'regions': [df[dims[2]].nunique()],  # Count months and regions.
    'sex_categories': [df['Sex'].nunique()], 'age_groups': [df['Age'].nunique()],  # Count sex and age categories.
    'combinations_per_month': [counts.iloc[0]], 'latest_month': [df['Month'].max()]  # Show monthly coverage and latest month.
}))


,months,regions,sex_categories,age_groups,combinations_per_month,latest_month
0,176,87,2,6,1044,2026-08-01


### Tidy regional measures

The workbook labels its geography **Labour market region (SA4): ASGS (2011)**. Preserve that definition and the published region names, including combined regions. `series_type = Modelled` identifies this source; it does not assert a seasonal adjustment status. MLF1 does not supply ABS time-series IDs, so no series ID is exported. Only the underlying Data 1 regional records are used; national and state totals from the presentation pivot are not imported. No totals, rates or additional populations are inferred from the pivot table.

In [10]:
# Turn MLF1's three measure columns into rows and label each regional series.
mlf = df.rename(columns={  # Give the dimension columns consistent names.
    'Month': 'date', 'Sex': 'sex', 'Age': 'age', dims[2]: 'region'  # Rename the published month, sex, age, and region fields.
}).melt(id_vars=['date', 'sex', 'age', 'region'], value_vars=measures, var_name='measure', value_name='value')  # Make one row per measure.
mlf['region_code'] = mlf['region'].str.extract(r'^(\d{3}) ')[0]  # Take the three-digit code from each region name.
assert mlf['region_code'].notna().all(), 'MLF1: region-code format changed'  # Require every region to have a code.
mlf['region'] = mlf['region'].str.replace(r'^\d{3} ', '', regex=True)  # Keep the readable region name separately.
mlf['measure'] = mlf['measure'].str.replace(" ('000)", '', regex=False)  # Remove the unit suffix from measure names.
mlf['unit'] = '000'  # Preserve the workbook's thousands unit.
mlf['series_type'] = 'Modelled'  # Mark the estimates as modelled, not seasonally adjusted.
mlf['dataset'] = 'MLF1'  # Identify these rows as coming from MLF1.
assert len(mlf) == len(df) * len(measures)  # Require one output row per input measure.
print(mlf.shape)  # Show the size of the reshaped table.
display(mlf[['date', 'region_code', 'region', 'sex', 'age', 'measure', 'value']].head())  # Preview the regional rows.


(551232, 10)


,date,region_code,region,sex,age,measure,value
0,2012-01-01,102,Central Coast,Males,15-24 years,Employed total,12.757092
1,2012-01-01,115,Sydney - Baulkham Hills and Hawkesbury,Males,15-24 years,Employed total,9.284133
2,2012-01-01,116,Sydney - Blacktown,Males,15-24 years,Employed total,12.403918
3,2012-01-01,117,Sydney - City and Inner South,Males,15-24 years,Employed total,11.087451
4,2012-01-01,118,Sydney - Eastern Suburbs,Males,15-24 years,Employed total,8.474928


## 5. Save three cleaned source tables

Each source workbook gets its own cleaned CSV named with the ABS table and reference month. Table 010 includes both national and state/territory series. Its export omits region code, geography level, geography basis and reference period; region identifies Australia or the state/territory. Output filenames identify each source table. Table 014 keeps only `region` for geography, omitting region code, geography level and geography basis. Its export also omits `reference_period`; the release month remains in the filename and source manifest. `sources.csv` records the ABS download and output filename for each table. It is written last, after all three cleaned CSVs, and marks a completed run.

The time-series exports retain unit, series type, data type and series ID. Frequency, collection month, series start/end, observation count and source table are omitted from both numbered-table exports; the source metadata is still read internally. Series headings remain split into the clean dimensions, without a separate description column. MLF1 exports its regional rows with region code and name labelled `sa4_code` and `sa4_region`. It omits series ID, geography level/basis, data type, frequency, collection month and series start/end. It also omits `source_tables`, `n_obs` and `reference_period`; the release month remains in the filename and source manifest.


In [ ]:
# Write one cleaned CSV per ABS workbook.
cols = ['date', 'series_id', 'measure', 'sex', 'age', 'region_code', 'region',
        'geography_level', 'geography_basis', 'series_type', 'data_type', 'unit', 'value']  # Define the shared output fields before source-specific column selection.
exports = {  # Keep each numbered table's cleaned rows separate.
    dataset.lower().replace(' ', '_'): frame.assign(reference_period=reference_period)[cols + ['reference_period']]
    for dataset, frame in ts.groupby('dataset', sort=False)
}
exports['table_010'] = exports['table_010'].drop(columns=['reference_period', 'geography_basis', 'geography_level', 'region_code'])  # Region identifies Australia or the state/territory.
exports['table_014'] = exports['table_014'].drop(columns=['region_code', 'geography_level', 'geography_basis', 'reference_period'])  # Keep only the region name for geography; the release month remains in the manifest.
mlf_cols = ['date', 'measure', 'sex', 'age', 'region_code', 'region', 'series_type', 'unit',
            'value']  # Keep only the requested MLF1 output fields.
exports['mlf1'] = (
    mlf[mlf_cols]
       .rename(columns={'region_code': 'sa4_code', 'region': 'sa4_region'})  # Name the published SA4 region explicitly.
)

for name, frame in exports.items():  # Save each source table with the ABS reference month in its filename.
    filename = f'{name}_{reference_period:%Y-%m}.csv'  # Example: table_010_2026-08.csv.
    frame.to_csv(run / filename, index=False, date_format='%Y-%m-%d')  # Write its cleaned rows.
    sources.loc[sources.index.str.lower().str.replace(' ', '_') == name, 'output_file'] = filename  # Link this CSV to its source workbook.

sources.to_csv(run / 'sources.csv')  # Mark the run complete after all three CSVs are saved.
print(f'Saved {len(exports)} cleaned CSV files for {reference_period:%B %Y}.')  # Show the completed export count.
display(sources[['download_file', 'output_file']])  # Preview the source-to-output mapping.
print(run)  # Show where this run was saved.

# Temporary previews of the exported tables.
with pd.option_context('display.max_columns', None):
    for name, frame in exports.items():
        print(f'\n{name}_{reference_period:%Y-%m}.csv — {frame.shape[0]:,} rows × {frame.shape[1]} columns')
        display(frame.head(5))
